## Conceptual Compression Retrivers

In [15]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
from langchain_openai import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.documents import Document

In [17]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [18]:
# Create a FAISS vector store from the documents
Embeddings = OpenAIEmbeddings(
    model = os.getenv('EMBEDDING_MODEL'),
    api_key=os.getenv('OPENAI_API_KEY')
)


In [19]:
from langchain_community.vectorstores import FAISS

In [20]:
vector_store = FAISS.from_documents(documents=docs,embedding=Embeddings)

In [21]:
base_retriver = vector_store.as_retriever(search_kwargs={'k':3})

## importing the langchain Conceputal Compression Retrivals

In [22]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [23]:
llm = ChatOpenAI(
    model = os.getenv('OPENAI_MODEL'),
    api_key=os.getenv('OPENAI_API_KEY'),
    temperature=0.3
)

In [24]:
compressor = LLMChainExtractor.from_llm(llm)

In [25]:
# Create the contextual compression retriever
conceptual_retriver = ContextualCompressionRetriever(
    base_retriever=base_retriver,
    base_compressor=compressor
)

In [26]:
# Query to the retrievers
qurey = 'what is photosynthesis?'
compressed_results = conceptual_retriver.invoke(qurey)

In [27]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
Photosynthesis does not occur in animal cells.

--- Result 3 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
